# 02 · The splits — country, region, city, provider

Every slice of this dataset holds the same four metrics and the same percentiles. This helps you to explore these splits to see how you could use them to ask questions of the data:

**How do internet measurements vary — between countries, within a country, or between providers?**

| Slice | One row per… | What you can learn |
|---|---|---|
| `by_country` | country | Where the country itself sits among others (notebook 01) |
| `by_country_subdivision1` | state / province | The range *across* one country's adminstrative districts / areas (states, provinces etc)  |
| `by_country_city` | city | How cities vary around the country |
| `by_country_asn` | provider | Provider differences — **compare carefully** |

> **About providers (ASNs):** these data are not cleaned, and small-sample rows can mislead.

## Setup — the same loader, plus one trick

We use the newest month that exists in **all four slices** this notebook needs. 

In [ ]:
# ── Setup: imports, manifest, catalog ─────────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# Plot style: light grid lines and a muted palette.
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

# Download the manifest, then build a table of every published file.
MANIFEST_URL = "https://measurementlab.net/data/stats/manifest.json"
manifest = requests.get(MANIFEST_URL, timeout=30).json()

records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    # cache/v1/{start_ts}/{end_ts}/{slice_name}/data.parquet
    if len(parts) == 6 and parts[5] == "data.parquet":
        records.append({
            "start": pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "end":   pd.to_datetime(parts[3], format="%Y%m%dT%H%M%SZ"),
            "slice": parts[4],
            "url":   meta["url"],
        })

catalog = (pd.DataFrame(records)
           .sort_values(["slice", "start"])
           .reset_index(drop=True))

# Helper: the direct download URL for one month of one slice.
def month_url(slice_name, start):
    # start looks like 'YYYY-MM-DD' (the first day of the month).
    row = catalog[(catalog["slice"] == slice_name) &
                  (catalog["start"] == pd.to_datetime(start))]
    if row.empty:
        raise ValueError(f"No {slice_name} file for {start}")
    return row.iloc[0]["url"]

# Helper: the newest month that has data for a slice.
def latest_month(slice_name):
    return catalog.loc[catalog["slice"] == slice_name, "start"].max().strftime("%Y-%m-%d")

print("Catalog loaded —", len(catalog), "files,",
      catalog["slice"].nunique(), "slices.")

In [ ]:
# ── Pick one month that every slice has ───────────────────────────────────────
# Slices are published at slightly different times, so take the newest month
# that exists in ALL of them — otherwise one plot would have no data.
SLICES = ["downloads_by_country", "downloads_by_country_subdivision1",
          "downloads_by_country_city", "downloads_by_country_asn"]


def latest_common_month(slice_names):
    months = None
    for s in slice_names:
        s_months = set(catalog.loc[catalog["slice"] == s, "start"])
        months = s_months if months is None else months & s_months
    return max(months).strftime("%Y-%m-%d")


MONTH = latest_common_month(SLICES)
print("Newest month present in all slices:", MONTH)

## Regions within a country

Pick a country: its states/provinces ranked by your chosen metric. The *range* between the fastest and slowest region is the internal geography of quality — compare a country with a tight range (regions roughly alike) against one with a wide range (a capital far ahead of the periphery).

> **Sample counts are important to consider here.** We've set a minimum of 500, so you may not see regions with very few tests.

In [ ]:
# ── Regions within a country ──────────────────────────────────────────────────
# The metrics we can explore in every slice: label → (column, download/upload).
SPLIT_METRICS = {
    "Download p50": ("download_p50", "downloads"),
    "Upload p50":   ("upload_p50",   "uploads"),
    "Latency p50":  ("latency_p50",  "downloads"),
    "Loss p50":     ("loss_p50",     "downloads"),
}
# Lower-is-better flags for latency and loss.
LOWER = {"Download p50": False, "Upload p50": False,
         "Latency p50": True, "Loss p50": True}

# Load this month's region data and list the countries that have any.
regions = pd.read_parquet(month_url("downloads_by_country_subdivision1", MONTH))
reg_countries = sorted(regions["country_code"].dropna().unique())

# Controls: country, metric, minimum test count, how many regions to show.
w_rc = widgets.Dropdown(options=reg_countries, value="US", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_rm = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_rmin = widgets.IntSlider(value=500, min=0, max=50000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
w_rn = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Show:",
                          layout=widgets.Layout(width="300px"))
out_r = widgets.Output()

# Redraw: keep regions that pass the test-count floor, rank them, plot.
def update_r(change=None):
    col, table = SPLIT_METRICS[w_rm.value]
    slice_name = f"{table}_by_country_subdivision1"
    data = pd.read_parquet(month_url(slice_name, MONTH))
    data = data[(data["country_code"] == w_rc.value) & (data["sample_count"] >= w_rmin.value)]
    if data.empty:
        with out_r:
            clear_output(wait=True)
            print("No regions pass the minimum test count — lower 'Min tests'.")
        return
    top = data.nsmallest(w_rn.value, col) if LOWER[w_rm.value] else data.nlargest(w_rn.value, col)
    top = top.sort_values(col, ascending=not LOWER[w_rm.value])
    with out_r:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.36)))
        ax.barh(top["subdivision1_name"], top[col])
        ax.set_xlabel(w_rm.value)
        ax.set_title(f"Regions of {w_rc.value} by {w_rm.value} — {MONTH}")
        plt.tight_layout(); plt.show()

for w in (w_rc, w_rm, w_rmin, w_rn):
    w.observe(update_r, "value")
display(widgets.VBox([widgets.HBox([w_rc, w_rm]), w_rmin, w_rn, out_r]))
update_r()

## Cities

Same pattern, finer grained geography.

In [ ]:
# ── Cities ────────────────────────────────────────────────────────────────────
# Load this month's city data, then the same pattern as regions, at finer
# granularity.
city = pd.read_parquet(month_url("downloads_by_country_city", MONTH))
city_countries = sorted(city["country_code"].dropna().unique())

# Controls: country, metric, minimum test count, how many cities to show.
w_cc = widgets.Dropdown(options=city_countries, value="BR", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_cm = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_cmin = widgets.IntSlider(value=1000, min=0, max=50000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
w_cn = widgets.IntSlider(value=15, min=5, max=40, step=5, description="Show:",
                          layout=widgets.Layout(width="300px"))
out_c = widgets.Output()

# Redraw: keep cities that pass the test-count floor, rank them, plot.
def update_c(change=None):
    col, table = SPLIT_METRICS[w_cm.value]
    slice_name = f"{table}_by_country_city"
    data = pd.read_parquet(month_url(slice_name, MONTH))
    data = data[(data["country_code"] == w_cc.value) & (data["sample_count"] >= w_cmin.value)]
    if data.empty:
        with out_c:
            clear_output(wait=True)
            print("No cities pass the minimum test count — lower 'Min tests'.")
        return
    top = data.nsmallest(w_cn.value, col) if LOWER[w_cm.value] else data.nlargest(w_cn.value, col)
    top = top.sort_values(col, ascending=not LOWER[w_cm.value])
    with out_c:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.36)))
        ax.barh(top["city"], top[col])
        ax.set_xlabel(w_cm.value)
        ax.set_title(f"Cities of {w_cc.value} by {w_cm.value} — {MONTH}")
        plt.tight_layout(); plt.show()

for w in (w_cc, w_cm, w_cmin, w_cn):
    w.observe(update_c, "value")
display(widgets.VBox([widgets.HBox([w_cc, w_cm]), w_cmin, w_cn, out_c]))
update_c()

## Providers (ASNs)

Regulators, journalists, and consumers often want provider information. Be careful with comparisons here, and make sure you have a high `sample_count` before you draw any conclusions with these data. 


In [ ]:
# ── Providers (ASNs) ──────────────────────────────────────────────────────────
# Load this month's provider data. The slice already includes provider names
# (as_name), so no lookup table is needed.
asn = pd.read_parquet(month_url("downloads_by_country_asn", MONTH))
asn_countries = sorted(asn["country_code"].dropna().unique())

# Controls: country, metric, minimum test count, how many providers to show.
# The sample floor starts high on purpose — small provider samples mislead.
w_ac = widgets.Dropdown(options=asn_countries, value="KE", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_am = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_amin = widgets.IntSlider(value=2000, min=0, max=100000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
w_an = widgets.IntSlider(value=15, min=5, max=40, step=5, description="Show:",
                          layout=widgets.Layout(width="300px"))
out_a = widgets.Output()

# Redraw: keep providers that pass the floor, rank them, label bars by name.
def update_a(change=None):
    col, table = SPLIT_METRICS[w_am.value]
    slice_name = f"{table}_by_country_asn"
    data = pd.read_parquet(month_url(slice_name, MONTH))
    data = data[(data["country_code"] == w_ac.value) & (data["sample_count"] >= w_amin.value)]
    if data.empty:
        with out_a:
            clear_output(wait=True)
            print("No providers pass the high minimum — that is information too. "
                  "Lower 'Min tests' only with care.")
        return
    top = data.nsmallest(w_an.value, col) if LOWER[w_am.value] else data.nlargest(w_an.value, col)
    top = top.sort_values(col, ascending=not LOWER[w_am.value]).copy()
    top["label"] = top.apply(lambda r: f"{r['as_name']} (AS{r['asn']})", axis=1)
    with out_a:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.42)))
        ax.barh(top["label"], top[col])
        ax.set_xlabel(w_am.value)
        ax.set_title(f"Providers in {w_ac.value} by {w_am.value} — {MONTH} "
                     f"(each ≥ {w_amin.value} tests)")
        plt.tight_layout(); plt.show()

for w in (w_ac, w_am, w_amin, w_an):
    w.observe(update_a, "value")
display(widgets.VBox([widgets.HBox([w_ac, w_am]), w_amin, w_an, out_a]))
update_a()